# End-to-End Machine Learning Pipeline
## Loan Default Prediction

**Dataset:** Finance Loan Default Dataset  
**Model:** Logistic Regression (Baseline)  
**Goal:** Predict whether a loan applicant will default

## Step 1: Load the Dataset

We load the dataset using pandas. The file uses a .xls extension but is stored as a CSV, so we use read_csv.

In [ ]:
# Import the pandas library for data manipulation
import pandas as pd

# Load the dataset — file uses .xls extension but is stored as a CSV
df = pd.read_csv('finance_loan_default_dataset.xls')

# Preview the first 5 rows to confirm it loaded correctly
print(df.head())

# Check the total number of rows and columns
print(df.shape)

## Step 2: Inspect the Dataset

We inspect the dataset to understand its structure, data types, missing values, and duplicates before making any changes.

In [ ]:
# Display all column names in the dataset
print(df.columns)

# Show data types and non-null counts for each column
print(df.info())

# Display summary statistics for numerical columns
print(df.describe())

# Count missing values in each column
print(df.isnull().sum())

# Check for any duplicate rows
print(df.duplicated().sum())

## Step 3: Clean the Data

We remove duplicate rows and fill missing values.  
Numerical columns are filled with the median. Categorical columns are filled with the most frequent value.

In [ ]:
# Check number of duplicate rows before cleaning
print("Duplicates before:", df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

# Confirm duplicates have been removed
print("Duplicates after:", df.duplicated().sum())

# Fill missing numerical values with the median of each column
numerical_columns = ['Age', 'Annual_Income', 'Credit_Score', 'Savings_Balance']
for col in numerical_columns:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with the most frequent value (mode)
categorical_columns = ['Employment_Status', 'Account_Type']
for col in categorical_columns:
    df[col] = df[col].fillna(df[col].mode()[0])

# Confirm there are no more missing values
print("Missing values after cleaning:")
print(df.isnull().sum())

## Step 4: Define Features and Target

The target variable is what the model predicts — Loan_Default.  
The feature columns are the inputs the model learns from.  
Applicant_ID is a row identifier with no predictive value and is excluded.

In [ ]:
# Define the feature columns the model will learn from
# Applicant_ID is excluded as it is just a row identifier
feature_columns = [
    'Age', 'Annual_Income', 'Credit_Score', 'Loan_Amount',
    'Loan_Term_Months', 'Existing_Debt', 'Late_Payments_Last_Year',
    'Savings_Balance', 'Debt_to_Income_Ratio',
    'Employment_Status', 'Account_Type', 'Has_Credit_Card'
]

# Assign features to X and target to y
X = df[feature_columns]
y = df['Loan_Default']

# Encode the target variable: No -> 0, Yes -> 1
y = y.map({'No': 0, 'Yes': 1})

# Confirm the shapes of X and y
print("X shape:", X.shape)
print("y shape:", y.shape)

# Check the class distribution of the target variable
print(y.value_counts())

## Step 5: Train/Test Split

We split the data into 80% training and 20% testing.  
stratify=y ensures both splits maintain the same class balance.

In [ ]:
# Import the train/test split function
from sklearn.model_selection import train_test_split

# Split the data — 80% for training, 20% for testing
# random_state=42 ensures reproducibility
# stratify=y maintains the same class ratio in both splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Confirm the size of each split
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## Step 6: Preprocessing

Numerical features are scaled using StandardScaler.  
Categorical features are encoded using OneHotEncoder.  
A ColumnTransformer applies both steps to the correct columns.  
The preprocessor is fitted on training data only and applied to the test set.

In [ ]:
# Import the necessary preprocessing tools
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Define which columns are numerical and which are categorical
numerical_features = [
    'Age', 'Annual_Income', 'Credit_Score', 'Loan_Amount',
    'Loan_Term_Months', 'Existing_Debt', 'Late_Payments_Last_Year',
    'Savings_Balance', 'Debt_to_Income_Ratio'
]

categorical_features = ['Employment_Status', 'Account_Type', 'Has_Credit_Card']

# Build the preprocessor:
# - StandardScaler normalises numerical features to the same scale
# - OneHotEncoder converts categorical features into binary columns
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numerical_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Fit the preprocessor on training data only, then apply to both sets
# This prevents data leakage from the test set
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

# Confirm the new shapes after preprocessing
print("X_train after preprocessing:", X_train_preprocessed.shape)
print("X_test after preprocessing:", X_test_preprocessed.shape)

## Step 7: Train the Model

We train a Logistic Regression model as the baseline classifier.

In [ ]:
# Import the Logistic Regression classifier
from sklearn.linear_model import LogisticRegression

# Initialise the model — max_iter=1000 ensures convergence
# random_state=42 makes results reproducible
model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model on the preprocessed training data
model.fit(X_train_preprocessed, y_train)

print("Model trained successfully!")

## Step 8: Evaluate the Model

We evaluate the model using accuracy, precision, recall, F1 score, a classification report, and a confusion matrix.

In [ ]:
# Import evaluation metrics and visualisation tools
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Use the trained model to make predictions on the test set
y_pred = model.predict(X_test_preprocessed)

# Print individual evaluation metrics
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

# Print the full classification report with per-class breakdown
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No Default', 'Default']))

# Plot the confusion matrix to visualise true vs predicted labels
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Default', 'Default']).plot()
plt.title("Confusion Matrix - Logistic Regression")
plt.show()

## Summary and Reflection

### What the model predicts
The model predicts whether a loan applicant will default on their loan — a binary outcome (Yes = 1, No = 0). This is a supervised binary classification problem.

### Dataset used
The Finance Loan Default Dataset containing 356 applicant records with 14 columns covering demographic, financial, and credit history information.

### Features and target selected
- **Target variable:** Loan_Default (1 = defaulted, 0 = did not default)
- **Numerical features (9):** Age, Annual_Income, Credit_Score, Loan_Amount, Loan_Term_Months, Existing_Debt, Late_Payments_Last_Year, Savings_Balance, Debt_to_Income_Ratio
- **Categorical features (3):** Employment_Status, Account_Type, Has_Credit_Card
- **Excluded:** Applicant_ID — a row identifier with no predictive value

### Results obtained
The Logistic Regression baseline model achieved 80% accuracy on the 20% held-out test set. The model performs well on the No Default class (recall = 0.94) but struggles with the Default class (recall = 0.39), meaning it misses many actual defaults.

### One limitation
**Class imbalance:** The dataset contains roughly 3x more non-default cases than default cases. The model is biased towards predicting No Default, which is dangerous in a financial setting where missing an actual default is far more costly than a false alarm. Techniques such as SMOTE oversampling, class-weight adjustment, or a more powerful model like Random Forest or XGBoost with threshold tuning would improve performance on the minority class.